# Proyecto de Machine Learning: Clasificación de GRD en Hospital El Pino

Este notebook refactoriza completamente el flujo de trabajo para abordar el problema correcto: **clasificación multiclase desbalanceada** del código GRD a partir de diagnósticos, procedimientos, edad y sexo.

La variable objetivo se construye extrayendo los **primeros 6 caracteres** de la columna `GRD`.


## 1. Limpieza de datos (Limpiar y filtrar)

Se importan librerías, se fija la semilla, se lee el archivo base `dataset_elpino.csv` tratando `-` como valor faltante, y se eliminan duplicados exactos junto con filas sin información esencial para el modelado.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


In [ ]:
DATA_PATH = Path("dataset_elpino.csv")
df_raw = pd.read_csv(DATA_PATH, sep=";", na_values=["-"], low_memory=False)

print(f"Dimensión original: {df_raw.shape}")
display(df_raw.head(3))


In [ ]:
diag_cols = [c for c in df_raw.columns if c.startswith("Diag ")]
proc_cols = [c for c in df_raw.columns if c.startswith("Proced ")]
age_col = next(c for c in df_raw.columns if "Edad en" in c)
sex_col = "Sexo (Desc)"
grd_col = "GRD"
main_diag_col = next(c for c in diag_cols if "Principal" in c)

df = df_raw.copy()

duplicados_antes = df.duplicated().sum()
df = df.drop_duplicates()

df["GRD_CODE"] = (
    df[grd_col]
    .astype(str)
    .str.strip()
    .replace("nan", np.nan)
    .str[:6]
)

df[age_col] = pd.to_numeric(df[age_col], errors="coerce")
df[sex_col] = df[sex_col].astype(str).str.strip().replace("nan", np.nan)
df[main_diag_col] = df[main_diag_col].astype(str).str.strip().replace("nan", np.nan)

filas_antes = len(df)
df = df.dropna(subset=["GRD_CODE", age_col, sex_col, main_diag_col])
df = df[df[age_col].between(0, 110)]

print(f"Duplicados exactos eliminados: {duplicados_antes}")
print(f"Filas eliminadas por datos esenciales faltantes o edad inválida: {filas_antes - len(df)}")
print(f"Dimensión tras limpieza: {df.shape}")


## 2. Preprocesamiento (Etiquetar datos y asignar valores numéricos)

Se crean variables de conteo (`n_diags`, `n_procs`), se binariza sexo, se codifica el diagnóstico principal con `LabelEncoder`, se arma `X` e `y`, y se realiza una división entrenamiento/prueba 80/20 con `stratify=y` después de remover clases con una sola observación.


In [ ]:
df["n_diags"] = df[diag_cols].notna().sum(axis=1)
df["n_procs"] = df[proc_cols].notna().sum(axis=1)

sexo_map = {
    "Mujer": 0,
    "Hombre": 1,
}
df["sexo_bin"] = df[sex_col].map(sexo_map)
df = df.dropna(subset=["sexo_bin"]).copy()

diag_encoder = LabelEncoder()
df["diag_principal_le"] = diag_encoder.fit_transform(df[main_diag_col].astype(str))

class_counts = df["GRD_CODE"].value_counts()
valid_classes = class_counts[class_counts > 1].index
df_model = df[df["GRD_CODE"].isin(valid_classes)].copy()

feature_cols = ["diag_principal_le", "n_diags", "n_procs", age_col, "sexo_bin"]
X = df_model[feature_cols].copy()
y = df_model["GRD_CODE"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Registros para modelado: {len(df_model):,}")
print(f"Clases originales: {df['GRD_CODE'].nunique()}")
print(f"Clases usadas tras remover singletons: {df_model['GRD_CODE'].nunique()}")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")


## 3. EDA (Análisis Exploratorio)

Se analiza la distribución de edad y se visualiza el fuerte desbalance de clases en los códigos GRD.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_model[age_col], bins=30, kde=True, ax=axes[0], color="#287271")
axes[0].set_title("Distribución de Edad")
axes[0].set_xlabel("Edad")

sns.boxplot(data=df_model, x=sex_col, y=age_col, ax=axes[1], palette="Set2")
axes[1].set_title("Edad por Sexo")
axes[1].set_xlabel("Sexo")
axes[1].set_ylabel("Edad")

plt.tight_layout()
plt.show()


In [ ]:
top20_grd = df_model["GRD_CODE"].value_counts().head(20)

plt.figure(figsize=(14, 6))
sns.barplot(x=top20_grd.index, y=top20_grd.values, palette="viridis")
plt.title("Top 20 GRD más frecuentes")
plt.xlabel("Código GRD (6 caracteres)")
plt.ylabel("Frecuencia")
plt.xticks(rotation=75)
plt.tight_layout()
plt.show()


## 4. Modelado (Clasificar y hacer predicciones)

Se entrena un `RandomForestClassifier` con `class_weight='balanced'` para abordar el desbalance multiclase.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=1,
)

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Modelo entrenado correctamente.")


## 5. Evaluación del Modelo (Exclusivo para Clasificación)

En esta sección se usan únicamente métricas válidas para clasificación: `Accuracy`, `Precision`, `Recall`, `F1-score`, `classification_report`, matriz de confusión y curvas ROC One-vs-Rest.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

metrics_df = pd.DataFrame(
    {
        "Métrica": ["Accuracy", "Precision (weighted)", "Recall (weighted)", "F1-score (weighted)"],
        "Valor": [accuracy, precision, recall, f1],
    }
)

display(metrics_df)


In [ ]:
top10_classes = df_model["GRD_CODE"].value_counts().head(10).index.tolist()
mask_top10 = y_test.isin(top10_classes)

report_top10 = classification_report(
    y_test[mask_top10],
    y_pred[mask_top10],
    labels=top10_classes,
    zero_division=0,
    output_dict=True,
)

report_top10_df = pd.DataFrame(report_top10).transpose()
display(report_top10_df)


In [ ]:
cm = confusion_matrix(y_test[mask_top10], y_pred[mask_top10], labels=top10_classes)

fig, ax = plt.subplots(figsize=(12, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=top10_classes)
disp.plot(ax=ax, cmap="Blues", xticks_rotation=75, colorbar=False)
ax.set_title("Matriz de Confusión - Top 10 clases más frecuentes")
plt.tight_layout()
plt.show()


In [ ]:
top5_classes = df_model["GRD_CODE"].value_counts().head(5).index.tolist()
proba_df = pd.DataFrame(rf_model.predict_proba(X_test), columns=rf_model.classes_)

plt.figure(figsize=(10, 7))

for cls in top5_classes:
    y_true_binary = (y_test == cls).astype(int)
    if y_true_binary.nunique() < 2:
        continue
    fpr, tpr, _ = roc_curve(y_true_binary, proba_df[cls])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, lw=2, label=f"{cls} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.title("Curvas ROC One-vs-Rest - Top 5 clases más frecuentes")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend()
plt.tight_layout()
plt.show()


## 6. Visualizaciones del Modelo

Se muestran la importancia de variables y una aproximación de "Loss vs Epochs" usando la evolución del `OOB Score` al incrementar el número de árboles.


In [ ]:
feature_importance = (
    pd.DataFrame(
        {
            "feature": feature_cols,
            "importance": rf_model.feature_importances_,
        }
    )
    .sort_values("importance", ascending=False)
)

plt.figure(figsize=(10, 5))
sns.barplot(data=feature_importance, x="importance", y="feature", palette="crest")
plt.title("Importancia de Variables")
plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.tight_layout()
plt.show()

display(feature_importance)


In [ ]:
tree_grid = [10, 50, 100, 200]
oob_scores = []

for n_trees in tree_grid:
    rf_oob = RandomForestClassifier(
        n_estimators=n_trees,
        random_state=42,
        class_weight="balanced",
        oob_score=True,
        bootstrap=True,
        n_jobs=1,
    )
    rf_oob.fit(X_train, y_train)
    oob_scores.append(rf_oob.oob_score_)

plt.figure(figsize=(8, 5))
plt.plot(tree_grid, oob_scores, marker="o", linewidth=2, color="#c1666b")
plt.title("OOB Score vs n_estimators")
plt.xlabel("Número de árboles")
plt.ylabel("OOB Score")
plt.xticks(tree_grid)
plt.tight_layout()
plt.show()

display(pd.DataFrame({"n_estimators": tree_grid, "oob_score": oob_scores}))


## 7. Discusión y Conclusiones

El problema tratado es multiclase y fuertemente desbalanceado, por lo que la evaluación debe centrarse en métricas de clasificación ponderadas y análisis por clase. El uso de `class_weight='balanced'` ayuda a compensar parcialmente el desbalance, aunque no lo elimina por completo.

La calidad final del modelo depende de la señal contenida en las variables disponibles. Aquí se incorporan variables clínicas resumidas y una codificación del diagnóstico principal, pero aún existe espacio para mejorar con mejor representación de diagnósticos/procedimientos y estrategias más avanzadas para clases raras.


In [ ]:
sample_idx = y_test.sample(1, random_state=42).index[0]
sample_features = X_test.loc[[sample_idx]].copy()
sample_real = y_test.loc[sample_idx]
sample_pred = rf_model.predict(sample_features)[0]

print("Ejemplo de Predicción")
display(sample_features)
print(f"GRD real: {sample_real}")
print(f"Predicción del modelo: {sample_pred}")
